# NeRF Data Exploration

This notebook explores the NeRF dataset structure, visualizes camera poses, and analyzes the data format.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import json
from PIL import Image
import imageio

# Add src to path
sys.path.insert(0, str(Path('..').resolve()))

from src.data import NeRFDataset

## 1. Load Dataset

In [ ]:
# Configure dataset path
DATA_DIR = '../data/nerf_synthetic/lego'  # Update with your path

# Load dataset
dataset = NeRFDataset(
    data_dir=DATA_DIR,
    split='train',
    image_scale=0.5,  # Use half resolution for faster exploration
    white_bg=True
)

print(f"Dataset size: {len(dataset)}")
print(f"Image size: {dataset.get_image_size()}")
print(f"Focal length: {dataset.focal_length:.2f}")
print(f"Near plane: {dataset.near:.3f}")
print(f"Far plane: {dataset.far:.3f}")

## 2. Visualize Sample Images

In [ ]:
# Display a few sample images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(min(8, len(dataset))):
    sample = dataset[i]
    image = sample['image'].numpy()
    
    axes[i].imshow(image)
    axes[i].set_title(f'Image {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Visualize Camera Poses in 3D

In [ ]:
# Extract camera positions
poses = np.array([dataset[i]['pose'].numpy() for i in range(len(dataset))])
camera_centers = poses[:, :3, 3]

# Visualize in 3D
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot camera centers
ax.scatter(camera_centers[:, 0], camera_centers[:, 1], camera_centers[:, 2], 
           c=range(len(camera_centers)), cmap='viridis', s=50)

# Plot camera orientations (where they're looking)
for i in range(0, len(camera_centers), 5):  # Sample every 5th camera
    center = camera_centers[i]
    # Camera looks in -z direction in camera space
    forward = poses[i, :3, 2]  # Third column is forward direction
    ax.quiver(center[0], center[1], center[2],
              forward[0], forward[1], forward[2],
              length=0.1, color='red', alpha=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Camera Poses in 3D Space')
plt.show()

print(f"Camera positions range:")
print(f"  X: [{camera_centers[:, 0].min():.3f}, {camera_centers[:, 0].max():.3f}]")
print(f"  Y: [{camera_centers[:, 1].min():.3f}, {camera_centers[:, 1].max():.3f}]")
print(f"  Z: [{camera_centers[:, 2].min():.3f}, {camera_centers[:, 2].max():.3f}]")

## 4. Analyze Image Statistics

In [ ]:
# Collect image statistics
all_images = []
for i in range(len(dataset)):
    image = dataset[i]['image'].numpy()
    all_images.append(image)

all_images = np.array(all_images)

print(f"Image shape: {all_images.shape}")
print(f"Pixel value range: [{all_images.min():.3f}, {all_images.max():.3f}]")
print(f"Mean pixel value: {all_images.mean():.3f}")
print(f"Std pixel value: {all_images.std():.3f}")

# Plot pixel value distribution
plt.figure(figsize=(10, 6))
plt.hist(all_images.flatten(), bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.title('Pixel Value Distribution')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Understand Ray Generation

In [ ]:
from src.rendering import generate_rays
import torch

# Get a sample image and pose
sample = dataset[0]
image = sample['image']
pose = sample['pose']
focal = sample['focal']
near = sample['near']
far = sample['far']

height, width = image.shape[:2]

# Generate rays
ray_origins, ray_directions = generate_rays(
    height, width, focal.item(), pose, near.item(), far.item()
)

print(f"Ray origins shape: {ray_origins.shape}")
print(f"Ray directions shape: {ray_directions.shape}")
print(f"Ray origin (center pixel): {ray_origins[height//2, width//2]}")
print(f"Ray direction (center pixel): {ray_directions[height//2, width//2]}")

# Visualize ray directions for center region
center_region = ray_directions[height//2-10:height//2+10, width//2-10:width//2+10]
print(f"\nCenter region ray directions shape: {center_region.shape}")
print(f"Ray directions are normalized: {torch.allclose(torch.norm(center_region, dim=-1), torch.ones_like(center_region[..., 0]))}")